# Career Path Prediction and Guidance System

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pickle

# Set seaborn style for visualizations
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Loading and Cleaning

In [ ]:
df = pd.read_csv('PS2_Dataset.csv')
print(f"Dataset Shape: {df.shape}")
df.info()

In [ ]:
print("Missing Values count:")
print(df.isnull().sum())

In [ ]:
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Visualizing the suggested job roles distribution
plt.figure(figsize=(10, 6))
sns.countplot(y='Suggested Job Role', data=df, order=df['Suggested Job Role'].value_counts().index, palette='viridis')
plt.title('Distribution of Suggested Job Roles')
plt.xlabel('Count')
plt.ylabel('Job Role')
plt.tight_layout()
plt.show()

In [ ]:
# Check coding skill rating distribution vs job roles
plt.figure(figsize=(12, 6))
sns.boxplot(x='coding skills rating', y='Suggested Job Role', data=df, palette='Set2')
plt.title('Coding Skills Rating by Suggested Job Role')
plt.xlabel('Coding Skills Rating (1-9)')
plt.ylabel('Suggested Job Role')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing and Feature Encoding

In [ ]:
# Copy dataframe to keep raw intact
processed_df = df.copy()

# Map binary columns
binary_cols = ['self-learning capability?', 'Extra-courses did', 'Taken inputs from seniors or elders', 'worked in teams ever?', 'Introvert']
for col in binary_cols:
    processed_df[col] = processed_df[col].str.lower().map({'yes': 1, 'no': 0})

# Map ordinal columns
skill_map = {'poor': 0, 'medium': 1, 'excellent': 2}
processed_df['reading and writing skills'] = processed_df['reading and writing skills'].str.lower().map(skill_map)
processed_df['memory capability score'] = processed_df['memory capability score'].str.lower().map(skill_map)

# Map technical vs management and work style
processed_df['Management or Technical'] = processed_df['Management or Technical'].str.lower().map({'technical': 1, 'management': 0})
processed_df['hard/smart worker'] = processed_df['hard/smart worker'].str.lower().map({'smart worker': 1, 'hard worker': 0})

# Map categorical columns using unique indices
categorical_cols = ['certifications', 'workshops', 'Interested subjects', 'interested career area ', 'Type of company want to settle in?', 'Interested Type of Books']
mappings = {}
for col in categorical_cols:
    unique_vals = sorted(list(processed_df[col].unique()))
    val_map = {val: idx for idx, val in enumerate(unique_vals)}
    processed_df[col] = processed_df[col].map(val_map)
    mappings[col] = val_map

# Map target Suggested Job Role
unique_targets = sorted(list(processed_df['Suggested Job Role'].unique()))
target_map = {val: idx for idx, val in enumerate(unique_targets)}
reverse_target_map = {idx: val for idx, val in enumerate(unique_targets)}
processed_df['Suggested Job Role'] = processed_df['Suggested Job Role'].map(target_map)
mappings['target'] = target_map
mappings['target_reverse'] = reverse_target_map

print("Preprocessing finished. Sample shape:", processed_df.shape)

In [ ]:
X = processed_df.drop(columns=['Suggested Job Role'])
y = processed_df['Suggested Job Role']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=101, stratify=y
)
print(f"Train set size: {X_train.shape[0]}, Test set size: {X_test.shape[0]}")

## 4. Model Training: Deep Learning Concept (MLP Classifier)

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    max_iter=300,
    random_state=101,
    early_stopping=True,
    n_iter_no_change=15
)
mlp.fit(X_train, y_train)
mlp_preds = mlp.predict(X_test)
print(f"Neural Network Accuracy: {accuracy_score(y_test, mlp_preds):.4f}")

## 5. Model Training: Ensemble Method (Random Forest Classifier)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_split=4,
    random_state=101
)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, rf_preds):.4f}")

## 6. Model Evaluation and Comparison

In [ ]:
rf_acc = accuracy_score(y_test, rf_preds)
mlp_acc = accuracy_score(y_test, mlp_preds)

print(f"Random Forest Accuracy: {rf_acc * 100:.2f}%")
print(f"Neural Network MLP Accuracy: {mlp_acc * 100:.2f}%")

best_model = rf if rf_acc >= mlp_acc else mlp
best_name = "Random Forest" if rf_acc >= mlp_acc else "Neural Network (MLP)"
print(f"\nChosen Production Model: {best_name}")

In [ ]:
print(f"Classification Report for {best_name}:")
target_names = [reverse_target_map[i] for i in range(len(unique_targets))]
print(classification_report(y_test, best_model.predict(X_test), target_names=target_names))

In [ ]:
# Feature Importance for Random Forest
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    features = X.columns
    
    plt.figure(figsize=(12, 6))
    sns.barplot(x=importances[indices], y=features[indices], palette='mako')
    plt.title('Feature Importances in Career Path Recommendation')
    plt.xlabel('Importance Value')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

## 7. Model Export for Deployment

In [ ]:
assets = {
    'model': best_model,
    'mappings': mappings,
    'features': list(X.columns),
    'rf_accuracy': rf_acc,
    'mlp_accuracy': mlp_acc,
    'best_model_name': best_name
}

with open('model_assets.pkl', 'wb') as f:
    pickle.dump(assets, f)
print("Model assets saved successfully to model_assets.pkl!")